In [ ]:
!pip -q install pandas
!pip -q install lxml
!pip -q install sentence-transformers
!pip -q install chromadb
!pip -q install tqdm
!pip -q install streamlit
!pip -q install pyngrok

In [ ]:
import os
import glob
import pandas as pd
import xml.etree.ElementTree as ET

from tqdm import tqdm

In [ ]:
!git clone https://github.com/abachaa/MedQuAD.git

In [ ]:
!ls MedQuAD

In [ ]:
xml_files = glob.glob("MedQuAD/**/*.xml", recursive=True)

print("Total XML files:", len(xml_files))

In [ ]:
questions = []
answers = []

for file in tqdm(xml_files):

    try:

        tree = ET.parse(file)

        root = tree.getroot()

        for qa in root.findall(".//QAPair"):

            question = qa.find("Question")

            answer = qa.find("Answer")

            if question is not None and answer is not None:

                questions.append(question.text)

                answers.append(answer.text)

    except:
        pass

In [ ]:
df = pd.DataFrame({

    "question": questions,

    "answer": answers

})

df.head()

In [ ]:
df = df.dropna()

df = df.drop_duplicates()

df.reset_index(drop=True, inplace=True)

print(df.shape)

In [ ]:
df.to_csv("medical_dataset.csv", index=False)

print("Dataset Saved")

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

In [ ]:
embeddings = embedding_model.encode(

    df["question"].tolist(),

    show_progress_bar=True

)

In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path="chroma_db"
)

collection = client.get_or_create_collection(
    "medical_qa"
)

In [ ]:
for i in tqdm(range(len(df))):

    question = str(df.iloc[i]["question"])
    answer = str(df.iloc[i]["answer"])

    if question == "nan" or answer == "nan":
        continue

    collection.add(

        ids=[str(i)],

        embeddings=[embeddings[i].tolist()],

        documents=[answer],

        metadatas=[

            {

                "question": question

            }

        ]

    )

In [ ]:
print(collection.count())

In [ ]:
def retrieve_answer(question):

    embedding = embedding_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[embedding],
        n_results=1
    )

    return results

In [ ]:
result = retrieve_answer("What is diabetes?")

print(result)

In [ ]:
DISEASES = [
    "diabetes",
    "cancer",
    "covid",
    "asthma",
    "hypertension",
    "arthritis",
    "malaria",
    "tuberculosis",
    "heart disease",
    "kidney disease",
    "anemia",
    "stroke",
    "migraine"
]

In [ ]:
SYMPTOMS = [
    "fever",
    "cough",
    "headache",
    "pain",
    "vomiting",
    "nausea",
    "fatigue",
    "rash",
    "dizziness",
    "chest pain",
    "sore throat",
    "shortness of breath",
    "weight loss"
]

In [ ]:
TREATMENTS = [
    "medicine",
    "medication",
    "insulin",
    "chemotherapy",
    "radiotherapy",
    "therapy",
    "exercise",
    "vaccination",
    "surgery",
    "dialysis",
    "physiotherapy",
    "antibiotics"
]

In [ ]:
def medical_entity_recognition(text):

    text = text.lower()

    diseases = []
    symptoms = []
    treatments = []

    for disease in DISEASES:
        if disease in text:
            diseases.append(disease)

    for symptom in SYMPTOMS:
        if symptom in text:
            symptoms.append(symptom)

    for treatment in TREATMENTS:
        if treatment in text:
            treatments.append(treatment)

    return {
        "Diseases": diseases,
        "Symptoms": symptoms,
        "Treatments": treatments
    }

In [ ]:
text = "I have fever, cough and diabetes."

entities = medical_entity_recognition(text)

print(entities)

In [ ]:
def medical_chat(question):

    entities = medical_entity_recognition(question)

    result = retrieve_answer(question)

    answer = result["documents"][0][0]

    return entities, answer

In [ ]:
question = "I have fever and cough."

entities, answer = medical_chat(question)

print("Question:")
print(question)

print("\nDetected Entities:")
print(entities)

print("\nRetrieved Answer:")
print(answer)

In [ ]:
while True:

    question = input("\nAsk a medical question (type 'exit' to quit): ")

    if question.lower() == "exit":
        break

    entities, answer = medical_chat(question)

    print("\nDetected Entities:")
    print(entities)

    print("\nAnswer:")
    print(answer)

    print("-" * 70)

In [ ]:
questions = [
    "What is asthma?",
    "How is cancer treated?",
    "What causes hypertension?",
    "What are symptoms of malaria?",
    "I have headache and fever."
]

for q in questions:

    entities, answer = medical_chat(q)

    print("=" * 70)

    print("Question:", q)

    print("Entities:", entities)

    print("Answer:", answer)

In [ ]:
!pip install -q streamlit pyngrok

In [ ]:
%%writefile app.py

import streamlit as st
import chromadb
from sentence_transformers import SentenceTransformer

# -----------------------------
# Load Embedding Model
# -----------------------------

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# -----------------------------
# Load ChromaDB
# -----------------------------

client = chromadb.PersistentClient(path="chroma_db")

collection = client.get_collection("medical_qa")

# -----------------------------
# Medical Dictionaries
# -----------------------------

DISEASES = [
    "diabetes","cancer","covid","asthma","hypertension",
    "malaria","arthritis","tuberculosis","heart disease",
    "kidney disease","anemia","stroke","migraine"
]

SYMPTOMS = [
    "fever","cough","headache","pain","vomiting",
    "nausea","fatigue","rash","dizziness",
    "chest pain","sore throat","shortness of breath"
]

TREATMENTS = [
    "medicine","medication","insulin","chemotherapy",
    "radiotherapy","therapy","exercise","vaccination",
    "surgery","dialysis","physiotherapy","antibiotics"
]

# -----------------------------
# Entity Recognition
# -----------------------------

def medical_entity_recognition(text):

    text = text.lower()

    diseases = []
    symptoms = []
    treatments = []

    for disease in DISEASES:
        if disease in text:
            diseases.append(disease)

    for symptom in SYMPTOMS:
        if symptom in text:
            symptoms.append(symptom)

    for treatment in TREATMENTS:
        if treatment in text:
            treatments.append(treatment)

    return diseases, symptoms, treatments

# -----------------------------
# Retrieve Answer
# -----------------------------

def retrieve_answer(question):

    embedding = embedding_model.encode(question).tolist()

    result = collection.query(
        query_embeddings=[embedding],
        n_results=1
    )

    return result["documents"][0][0]

# -----------------------------
# Streamlit UI
# -----------------------------

st.set_page_config(page_title="Medical Q&A Chatbot")

st.title("🏥 Medical Q&A Chatbot")

st.write("Ask any medical question from the MedQuAD dataset.")

question = st.text_input("Enter your medical question")

if st.button("Get Answer"):

    diseases, symptoms, treatments = medical_entity_recognition(question)

    answer = retrieve_answer(question)

    st.subheader("Detected Medical Entities")

    st.write("Diseases:", diseases)

    st.write("Symptoms:", symptoms)

    st.write("Treatments:", treatments)

    st.subheader("Retrieved Answer")

    st.success(answer)

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501